In [ ]:
# In[1]:

import pandas as pd
import pytz

# reuse variables where possible
tz = pytz.timezone('Asia/Shanghai')

# load files
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# ensure timestamp is integer
for df in (metric, trace, log, error_logs):
    df['timestamp'] = df['timestamp'].astype('int64')

# create timezone-aware datetime in Asia/Shanghai by interpreting epoch as UTC then converting
for df in (metric, trace, log, error_logs):
    df['dt_sh'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define window in Asia/Shanghai (inclusive)
window_start = pd.Timestamp('2021-03-12 10:30:00', tz=tz)
window_end = pd.Timestamp('2021-03-12 11:00:00', tz=tz)

# ---------- metric aggregations (full series for p95/p5) ----------
metric_full = metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    total_count='size',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

metric_win = metric[(metric['dt_sh'] >= window_start) & (metric['dt_sh'] <= window_end)]
metric_win_agg = metric_win.groupby(['cmdb_id', 'kpi_name'])['timestamp'].agg(
    count_in_window='size',
    earliest_in_window='min',
    latest_in_window='max'
).reset_index()

metric_comb = pd.merge(metric_full, metric_win_agg, how='left',
                       left_on=['cmdb_id', 'kpi_name'], right_on=['cmdb_id', 'kpi_name'])
metric_comb['count_in_window'] = metric_comb['count_in_window'].fillna(0).astype(int)
metric_comb['earliest_in_window'] = metric_comb['earliest_in_window'].where(metric_comb['earliest_in_window'].notna(), pd.NA)
metric_comb['latest_in_window'] = metric_comb['latest_in_window'].where(metric_comb['latest_in_window'].notna(), pd.NA)
metric_comb['source'] = 'metric'
metric_comb = metric_comb.rename(columns={'kpi_name': 'name'})

# ---------- trace aggregations ----------
trace_full = trace.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    total_count='size',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

trace_win = trace[(trace['dt_sh'] >= window_start) & (trace['dt_sh'] <= window_end)]
trace_win_agg = trace_win.groupby(['cmdb_id', 'trace_name'])['timestamp'].agg(
    count_in_window='size',
    earliest_in_window='min',
    latest_in_window='max'
).reset_index()

trace_comb = pd.merge(trace_full, trace_win_agg, how='left',
                      left_on=['cmdb_id', 'trace_name'], right_on=['cmdb_id', 'trace_name'])
trace_comb['count_in_window'] = trace_comb['count_in_window'].fillna(0).astype(int)
trace_comb['earliest_in_window'] = trace_comb['earliest_in_window'].where(trace_comb['earliest_in_window'].notna(), pd.NA)
trace_comb['latest_in_window'] = trace_comb['latest_in_window'].where(trace_comb['latest_in_window'].notna(), pd.NA)
trace_comb['source'] = 'trace'
trace_comb = trace_comb.rename(columns={'trace_name': 'name'})

# ---------- log aggregations ----------
log_full = log.groupby(['cmdb_id', 'log_name'])['value'].agg(
    total_count='size',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

log_win = log[(log['dt_sh'] >= window_start) & (log['dt_sh'] <= window_end)]
log_win_agg = log_win.groupby(['cmdb_id', 'log_name'])['timestamp'].agg(
    count_in_window='size',
    earliest_in_window='min',
    latest_in_window='max'
).reset_index()

log_comb = pd.merge(log_full, log_win_agg, how='left',
                    left_on=['cmdb_id', 'log_name'], right_on=['cmdb_id', 'log_name'])
log_comb['count_in_window'] = log_comb['count_in_window'].fillna(0).astype(int)
log_comb['earliest_in_window'] = log_comb['earliest_in_window'].where(log_comb['earliest_in_window'].notna(), pd.NA)
log_comb['latest_in_window'] = log_comb['latest_in_window'].where(log_comb['latest_in_window'].notna(), pd.NA)
log_comb['source'] = 'log'
log_comb = log_comb.rename(columns={'log_name': 'name'})

# ---------- error_logs aggregations ----------
error_full = error_logs.groupby(['cmdb_id'])['timestamp'].agg(
    total_count='size',
    earliest_in_full='min',
    latest_in_full='max'
).reset_index()

error_win = error_logs[(error_logs['dt_sh'] >= window_start) & (error_logs['dt_sh'] <= window_end)]
error_win_agg = error_win.groupby(['cmdb_id'])['timestamp'].agg(
    count_in_window='size',
    earliest_in_window='min',
    latest_in_window='max'
).reset_index()

error_comb = pd.merge(error_full, error_win_agg, how='left', on='cmdb_id')
error_comb['count_in_window'] = error_comb['count_in_window'].fillna(0).astype(int)
error_comb['earliest_in_window'] = error_comb['earliest_in_window'].where(error_comb['earliest_in_window'].notna(), pd.NA)
error_comb['latest_in_window'] = error_comb['latest_in_window'].where(error_comb['latest_in_window'].notna(), pd.NA)
error_comb['source'] = 'error_logs'
error_comb['name'] = ''  # empty name for error_logs
# error_logs have no p95/p5
error_comb['p95'] = pd.NA
error_comb['p5'] = pd.NA
# align columns to match others
error_comb = error_comb.rename(columns={'total_count': 'total_count'})

# ---------- unify columns ----------
# select and standardize columns for metric/trace/log
common_cols = ['source', 'cmdb_id', 'name', 'total_count', 'p95', 'p5', 'count_in_window', 'earliest_in_window', 'latest_in_window']

metric_out = metric_comb[['source', 'cmdb_id', 'name', 'total_count', 'p95', 'p5', 'count_in_window', 'earliest_in_window', 'latest_in_window']]
trace_out = trace_comb[['source', 'cmdb_id', 'name', 'total_count', 'p95', 'p5', 'count_in_window', 'earliest_in_window', 'latest_in_window']]
log_out = log_comb[['source', 'cmdb_id', 'name', 'total_count', 'p95', 'p5', 'count_in_window', 'earliest_in_window', 'latest_in_window']]
error_out = error_comb[['source', 'cmdb_id', 'name', 'total_count', 'p95', 'p5', 'count_in_window', 'earliest_in_window', 'latest_in_window']]

combined = pd.concat([metric_out, trace_out, log_out, error_out], ignore_index=True, sort=False)

# ensure types: p95/p5 numeric or NA; count_in_window int; earliest/latest possibly NA or int
combined['count_in_window'] = combined['count_in_window'].astype(int)
# sort by count_in_window descending, then total_count descending for tie-break
combined_sorted = combined.sort_values(by=['count_in_window', 'total_count'], ascending=[False, False]).reset_index(drop=True)

# limit to top 50 rows
result_df_top = combined_sorted.head(50)

# display compact result
result_df_top

```
Out[1]:
```
```python
# Summarize the previously computed result_df_top in plain English.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

df = result_df_top  # reuse existing variable from previous execution

# basic counts
total_rows = int(df.shape[0])
source_counts = df['source'].value_counts().to_dict()

# top 5 series by count_in_window (tie-broken by total_count)
top5 = df.sort_values(['count_in_window', 'total_count'], ascending=[False, False]).head(5)[
    ['source', 'cmdb_id', 'name', 'total_count', 'count_in_window']
].to_dict('records')

# Build a compact plain-English summary
lines = []
lines.append(f"Summary of aggregated telemetry (top {total_rows} rows shown):")
lines.append(f"- Time window examined (inclusive, Asia/Shanghai): 2021-03-12 10:30:00 to 2021-03-12 11:00:00")
lines.append(f"- Rows returned: {total_rows}")
lines.append("- Rows by source: " + ", ".join(f"{k}: {v}" for k, v in source_counts.items()))
lines.append("- Top 5 series by points inside the window (source, cmdb_id, name, total_count, count_in_window):")
for r in top5:
    name_display = r['name'] if r['name'] != '' else '(empty name)'
    lines.append(f"  • {r['source']}, {r['cmdb_id']}, {name_display}, total_count={r['total_count']}, count_in_window={r['count_in_window']}")

# Note on thresholds
lines.append("- Note: p95 and p5 values were computed using the full available series (not recomputed on the filtered window).")
summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source  cmdb_id                                            name  total_count           p95            p5  count_in_window  earliest_in_window  latest_in_window
0   metric     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryMax           31  2.040136e+09  2.040136e+09               31          1615516200        1615518000
1   metric     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed           31  7.812230e+08  1.152546e+08               31          1615516200        1615518000
2   metric     IG01     JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed           31  5.789193e+07  5.789193e+07               31          1615516200        1615518000
3   metric     IG01        JVM-Threads_7778_JVM_ThreadCount_Threads           31  7.250000e+01  6.700000e+01               31          1615516200        1615518000
4   metric     IG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00  0.000000e+00               31          1615516200        1615518000
..     ...      ...                                             ...          ...           ...           ...              ...                 ...               ...
45  metric  Redis01                      OSLinux-CPU_CPU_CPUSysTime           31  2.499316e+00  1.298673e+00               31          1615516200        1615518000
46  metric  Redis01                     OSLinux-CPU_CPU_CPUUserTime           31  2.523784e+01  2.506993e+01               31          1615516200        1615518000
47  metric  Redis01                          OSLinux-CPU_CPU_CPUWio           31  2.456500e-01  6.235000e-03               31          1615516200        1615518000
48  metric  Redis01                     OSLinux-CPU_CPU_CPUidleutil           31  7.259325e+01  7.050227e+01               31          1615516200        1615518000
49  metric  Redis01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00  0.000000e+00               31          1615516200        1615518000

[50 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing metric DataFrame and tz variable from previous execution.
import pandas as pd
import pytz

# Ensure metric is present
_ = metric  # reuse

# 1) Compute global p95 per (cmdb_id, kpi_name) using the full series
p95_df = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95')

# 2) Filter metric to the inclusive window [1615516200, 1615518000]
win_start_ts = 1615516200
win_end_ts = 1615518000
metric_win = metric[(metric['timestamp'] >= win_start_ts) & (metric['timestamp'] <= win_end_ts)].copy()

# 3) Join filtered points with per-group p95
metric_win = metric_win.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Compute aggregates per (cmdb_id, kpi_name) within the window
# Mark anomalies: value >= p95 (per instruction)
metric_win['is_anom'] = metric_win['value'] >= metric_win['p95']

agg_funcs = {
    'timestamp': ['size', lambda x: int(x.max())],  # size -> count_in_window, also keep a placeholder for max ts if needed
    'value': ['max'],
    'is_anom': ['sum']
}
# We'll compute earliest anomaly timestamp separately
grouped = metric_win.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    count_in_window = ('timestamp', 'size'),
    max_value_in_window = ('value', 'max'),
    anomalies_count = ('is_anom', 'sum'),
    p95 = ('p95', 'first')
).reset_index()

# earliest_anomaly_ts: min timestamp among points with is_anom True; null if none
anom_ts = metric_win[metric_win['is_anom']].groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp'].min().reset_index(name='earliest_anomaly_ts')

# Join earliest anomaly ts
grouped = grouped.merge(anom_ts, on=['cmdb_id', 'kpi_name'], how='left')

# Compute severity = max_value_in_window / p95 if p95 > 0 else null
def compute_severity(row):
    p95v = row['p95']
    if pd.isna(p95v) or p95v == 0:
        return pd.NA
    return row['max_value_in_window'] / p95v

grouped['severity'] = grouped.apply(compute_severity, axis=1)

# Keep only rows where anomalies_count > 0
anomalies = grouped[grouped['anomalies_count'] > 0].copy()

# Ensure types and ordering
anomalies['earliest_anomaly_ts'] = anomalies['earliest_anomaly_ts'].where(anomalies['earliest_anomaly_ts'].notna(), pd.NA).astype("Int64")
anomalies['count_in_window'] = anomalies['count_in_window'].astype(int)
anomalies['anomalies_count'] = anomalies['anomalies_count'].astype(int)

# 5) Sort by earliest_anomaly_ts ascending (earliest first), then by severity descending. Limit to top 20.
anomalies_sorted = anomalies.sort_values(by=['earliest_anomaly_ts', 'severity'], ascending=[True, False]).head(20)

# Select and order requested columns
result_cols = ['cmdb_id', 'kpi_name', 'count_in_window', 'anomalies_count', 'earliest_anomaly_ts', 'max_value_in_window', 'p95', 'severity']
anomalies_top20 = anomalies_sorted[result_cols].reset_index(drop=True)

# Single-line summary: how many distinct metric series had anomalies in the window
distinct_series_with_anomalies = int(anomalies.shape[0])
summary = f"{distinct_series_with_anomalies} distinct metric series had anomalies in the window [{win_start_ts} - {win_end_ts}]."

# Display compact outputs: the top table and the one-line summary
anomalies_top20, summary

```
Out[2]:
```
```python
# Reuse existing variables from prior executions to produce a plain-English summary and a compact top table.
# Display: a one-line plain-English summary and the top anomalous metric series (compact columns).
import pandas as pd

# determine number of distinct metric series with anomalies (try to reuse existing 'anomalies' or 'anomalies_top20')
if 'anomalies' in globals():
    num_series = int(anomalies.shape[0])
elif 'anomalies_top20' in globals():
    # anomalies_top20 contains only top 20; fall back to parsing summary if available
    if 'summary' in globals():
        # extract leading integer if possible
        import re
        m = re.match(r"(\d+)", summary)
        num_series = int(m.group(1)) if m else int(anomalies_top20.shape[0])
    else:
        num_series = int(anomalies_top20.shape[0])
else:
    num_series = 0

summary_line = (f"{num_series} distinct metric series had anomalies in the window "
                f"[1615516200 - 1615518000] (inclusive, UTC+8).")

# Prepare a compact top table: requested columns, limited to top 20 (already prepared as anomalies_top20)
if 'anomalies_top20' in globals():
    top_table = anomalies_top20[[
        'cmdb_id', 'kpi_name', 'count_in_window', 'anomalies_count',
        'earliest_anomaly_ts', 'max_value_in_window', 'p95', 'severity'
    ]]
else:
    top_table = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'count_in_window', 'anomalies_count',
        'earliest_anomaly_ts', 'max_value_in_window', 'p95', 'severity'
    ])

# Display the compact table and the single-line summary
top_table, summary_line
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  count_in_window  anomalies_count  earliest_anomaly_ts  max_value_in_window         p95  severity
0       apache02                             OSLinux-CPU_CPU_CPUWio               29                2           1615516200             1.504900    0.631920  2.381474
1           IG01                             OSLinux-CPU_CPU_CPUWio               29                2           1615516200             2.636900    1.121440  2.351352
2       Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31                2           1615516200            39.000000   17.000000  2.294118
3           IG02                            OSLinux-CPU_CPU_CPULoad               31                2           1615516200             2.540000    1.465000  1.733788
4           MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               11                1           1615516200             0.200000    0.125000       1.6
5           MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                8                1           1615516200             0.300000    0.195000  1.538462
6           MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               11                1           1615516200             6.000000    3.933350  1.525417
7       Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31                2           1615516200            29.000000   20.500000  1.414634
8        Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               30               30           1615516200             0.400000    0.300000  1.333333
9   ServiceTest8                                            app.cnt               31                2           1615516200           442.000000  334.500000  1.321375
10  ServiceTest7                                            app.mrt               31                2           1615516200             8.406230    6.755365  1.244378
11  ServiceTest6                                            app.mrt               31                2           1615516200             8.160980    6.677535  1.222155
12      Tomcat01                             OSLinux-CPU_CPU_CPUWio               18                1           1615516200             0.677700    0.561250  1.207483
13      apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31               31           1615516200             0.600000    0.500000       1.2
14      apache01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps               16                1           1615516200             0.833300    0.695825  1.197571
15      Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31                2           1615516200            26.000000   22.000000  1.181818
16      apache01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite               16                1           1615516200            11.466700    9.866650  1.162168
17       Mysql02                    OSLinux-CPU_CPU-3_SingleCpuUtil               11                1           1615516200            49.418600   42.857900   1.15308
18       Mysql01                             OSLinux-CPU_CPU_CPUWio               31                2           1615516200            15.743743   13.992050  1.125192
19       Mysql01                            OSLinux-CPU_CPU_CPULoad               31                2           1615516200             1.190000    1.105000  1.076923, '1703 distinct metric series had anomalies in the window [1615516200 - 1615518000].')```
```

In [ ]:
# In[3]:

# Aggregate anomalies at component (cmdb_id) level using existing 'anomalies' DataFrame.
import pandas as pd

# Reuse 'anomalies' from prior execution; if missing, create empty placeholder
if 'anomalies' not in globals():
    anomalies = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'count_in_window', 'anomalies_count',
        'earliest_anomaly_ts', 'max_value_in_window', 'p95', 'severity'
    ])

# Ensure we operate only on rows with anomalies_count > 0
anom = anomalies[anomalies['anomalies_count'] > 0].copy()

# Normalize types for safe aggregations
if not anom.empty:
    # earliest_anomaly_ts may be nullable Int64; convert to numeric for min
    anom['earliest_anomaly_ts'] = pd.to_numeric(anom['earliest_anomaly_ts'], errors='coerce').astype('Int64')
    anom['anomalies_count'] = anom['anomalies_count'].astype(int)
    # severity numeric or NA
    anom['severity'] = pd.to_numeric(anom['severity'], errors='coerce')

# 1) Component-level aggregations
comp_agg = anom.groupby('cmdb_id', observed=True).agg(
    num_series_with_anomaly = ('kpi_name', 'nunique'),
    total_anomaly_points = ('anomalies_count', 'sum'),
    earliest_anomaly_ts = ('earliest_anomaly_ts', 'min'),
    max_severity = ('severity', 'max')
).reset_index()

# 2) Identify representative KPI per component: the KPI with max_severity; ties broken by earliest_anomaly_ts
if not anom.empty:
    # merge max_severity to anomalies
    anom_with_max = anom.merge(comp_agg[['cmdb_id', 'max_severity']], on='cmdb_id', how='left')
    # filter to rows matching max_severity (handle NaNs: only keep exact matches)
    # Use boolean mask taking into account possible NaNs
    mask = anom_with_max['severity'].notna() & anom_with_max['max_severity'].notna() & (anom_with_max['severity'] == anom_with_max['max_severity'])
    tied = anom_with_max[mask].copy()
    # For components where max_severity is NaN (all severities NaN), tied will be empty; handle by selecting first KPI per cmdb_id
    # For tied rows, pick the one with earliest earliest_anomaly_ts; if earliest_anomaly_ts equal, pick the first occurrence
    tied_sorted = tied.sort_values(['cmdb_id', 'earliest_anomaly_ts'], ascending=[True, True])
    rep = tied_sorted.drop_duplicates(subset=['cmdb_id'], keep='first')[['cmdb_id', 'kpi_name', 'severity', 'earliest_anomaly_ts']]
    rep = rep.rename(columns={
        'kpi_name': 'top_kpi',
        'severity': 'top_kpi_severity',
        'earliest_anomaly_ts': 'top_kpi_earliest_ts'
    })

    # For components where max_severity is NA or no matching tied rows, fallback: choose KPI with earliest anomaly
    missing_cmdbs = set(comp_agg['cmdb_id']) - set(rep['cmdb_id'])
    if missing_cmdbs:
        fallback = anom[anom['cmdb_id'].isin(missing_cmdbs)].copy()
        # pick KPI with earliest anomaly_ts; tie-breaker by highest anomalies_count then first occurrence
        fallback_sorted = fallback.sort_values(['cmdb_id', 'earliest_anomaly_ts', 'anomalies_count'], ascending=[True, True, False])
        fb = fallback_sorted.drop_duplicates(subset=['cmdb_id'], keep='first')[['cmdb_id', 'kpi_name', 'severity', 'earliest_anomaly_ts']]
        fb = fb.rename(columns={
            'kpi_name': 'top_kpi',
            'severity': 'top_kpi_severity',
            'earliest_anomaly_ts': 'top_kpi_earliest_ts'
        })
        # combine rep and fallback
        rep = pd.concat([rep, fb], ignore_index=True, sort=False)

    # merge representative KPI info into component aggregation
    comp_agg = comp_agg.merge(rep, on='cmdb_id', how='left')
else:
    # empty results
    comp_agg['top_kpi'] = pd.NA
    comp_agg['top_kpi_severity'] = pd.NA
    comp_agg['top_kpi_earliest_ts'] = pd.NA

# 3) Sort by max_severity desc, then total_anomaly_points desc, limit to top 20
comp_agg_sorted = comp_agg.sort_values(by=['max_severity', 'total_anomaly_points'], ascending=[False, False]).head(20).reset_index(drop=True)

# Select and order required columns
output_cols = [
    'cmdb_id', 'num_series_with_anomaly', 'total_anomaly_points',
    'earliest_anomaly_ts', 'max_severity', 'top_kpi', 'top_kpi_severity', 'top_kpi_earliest_ts'
]
comp_agg_top20 = comp_agg_sorted[output_cols]

# 4) One-line summary: number of distinct components with at least one anomalous metric series
num_components_with_anomaly = int(comp_agg.shape[0])
summary_line = f"{num_components_with_anomaly} distinct components had at least one anomalous metric series in the window."

# Display compact results (DataFrame and single-line summary)
comp_agg_top20, summary_line

```
Out[3]:
```
```python
# Produce a plain-English summary string of the component-level anomaly aggregation results.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse comp_agg_top20 and comp_agg from previous execution
df = comp_agg_top20 if 'comp_agg_top20' in globals() else pd.DataFrame()
total_components = int(comp_agg.shape[0]) if 'comp_agg' in globals() else 0

lines = []
lines.append(f"{total_components} distinct components had at least one anomalous metric series in the window.")

if not df.empty:
    lines.append("Top components by max_severity (showing up to 5):")
    top5 = df.head(5)
    for i, row in top5.iterrows():
        cmdb = row['cmdb_id']
        num_series = int(row['num_series_with_anomaly'])
        total_points = int(row['total_anomaly_points'])
        max_sev = row['max_severity']
        top_kpi = row['top_kpi'] if pd.notna(row['top_kpi']) else '(none)'
        top_kpi_sev = row['top_kpi_severity']
        ts = row['top_kpi_earliest_ts']
        if pd.isna(ts):
            ts_str = 'N/A'
        else:
            ts_str = pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        lines.append(f"{i+1}. {cmdb}: max_severity={max_sev:g}, series={num_series}, points={total_points}; top_kpi=\"{top_kpi}\" (severity={top_kpi_sev:g}, earliest={ts_str})")
else:
    lines.append("No component-level anomaly rows to display.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  num_series_with_anomaly  total_anomaly_points  earliest_anomaly_ts  max_severity                                            top_kpi  top_kpi_severity  top_kpi_earliest_ts
0            MG02                       91                   557           1615516200     20.000000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         20.000000           1615516500
1         Mysql01                      226                  2167           1615516200     10.000000        Mysql-MySQL_3306_Innodb data pending writes         10.000000           1615516440
2        Tomcat03                      109                   931           1615516200      4.000000       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          4.000000           1615517700
3        apache02                       82                   616           1615516200      4.000000  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...          4.000000           1615516560
4        apache01                       81                   605           1615516200      4.000000    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps          4.000000           1615516680
5            IG02                       89                   607           1615516200      3.333333    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead          3.333333           1615517940
6         Mysql02                      225                  2107           1615516200      3.333333     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps          3.333333           1615516380
7         Redis02                      109                  1349           1615516200      3.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...          3.333333           1615516740
8            IG01                       90                   614           1615516200      2.832335                    OSLinux-CPU_CPU-0_SingleCpuidle          2.832335           1615516260
9        Tomcat04                      111                   972           1615516200      2.777778       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          2.777778           1615517880
10           MG01                       92                   563           1615516200      2.773333       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          2.773333           1615517640
11        Redis01                      114                  1321           1615516200      2.559528                             OSLinux-CPU_CPU_CPUWio          2.559528           1615516260
12  ServiceTest10                        4                    62           1615516200      1.798609                                            app.mrt          1.798609           1615517640
13       Tomcat02                      108                   946           1615516200      1.714332                             OSLinux-CPU_CPU_CPUWio          1.714332           1615516260
14       Tomcat01                      108                   879           1615516200      1.562500     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps          1.562500           1615517280
15   ServiceTest7                        4                    50           1615516200      1.515152                                            app.cnt          1.515152           1615517040
16   ServiceTest4                        4                    44           1615516200      1.493230                                            app.cnt          1.493230           1615516860
17   ServiceTest9                        4                    51           1615516200      1.360248                                            app.cnt          1.360248           1615517280
18   ServiceTest8                        4                    62           1615516200      1.321375                                            app.cnt          1.321375           1615516200
19   ServiceTest5                        4                    65           1615516200      1.316752                                            app.mrt          1.316752           1615517640, '29 distinct components had at least one anomalous metric series in the window.')```
```

In [ ]:
# In[4]:

# Analyze trace anomalies for specified candidate components (fixed aggregation implementation).
import pandas as pd
import pytz

# Reuse existing trace DataFrame if available; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')
candidates = ['MG02', 'Mysql01', 'Tomcat03', 'apache02', 'apache01']
win_start_ts = 1615516200
win_end_ts = 1615518000

# 1) Full-series p95 and p5 per (cmdb_id, trace_name)
pctl = trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter trace to the inclusive window and candidate components
trace_win = trace[
    (trace['timestamp'] >= win_start_ts) &
    (trace['timestamp'] <= win_end_ts) &
    (trace['cmdb_id'].isin(candidates))
].copy()

# If no points, return empty result and summary
if trace_win.empty:
    cols = ['cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
            'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
            'p5','severity_low']
    result_df = pd.DataFrame(columns=cols)
    summary_line = "0 distinct trace series had anomalies among the candidate components in the window. Components with anomalies: []"
    result_df, summary_line
else:
    # 3) Join p95/p5
    trace_win = trace_win.merge(pctl, on=['cmdb_id', 'trace_name'], how='left')

    # flags for anomalies
    trace_win['is_high'] = trace_win['value'] >= trace_win['p95']
    trace_win['is_low'] = trace_win['value'] <= trace_win['p5']

    # 3/4) Aggregate per (cmdb_id, trace_name)
    agg = trace_win.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
        count_in_window = ('timestamp', 'size'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        anomalies_count_high = ('is_high', 'sum'),
        anomalies_count_low  = ('is_low', 'sum'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    ).reset_index()

    # earliest anomaly timestamps computed from filtered subsets
    high_earliest = trace_win[trace_win['is_high']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_ts_high')
    low_earliest  = trace_win[trace_win['is_low']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_ts_low')

    agg = agg.merge(high_earliest, on=['cmdb_id','trace_name'], how='left')
    agg = agg.merge(low_earliest, on=['cmdb_id','trace_name'], how='left')

    # ensure integer counts
    agg['anomalies_count_high'] = agg['anomalies_count_high'].astype(int)
    agg['anomalies_count_low']  = agg['anomalies_count_low'].astype(int)
    agg['count_in_window'] = agg['count_in_window'].astype(int)

    # compute severities
    def sev_high(row):
        p95v = row['p95']
        if pd.isna(p95v) or p95v == 0:
            return pd.NA
        return row['max_value_in_window'] / p95v

    def sev_low(row):
        p5v = row['p5']
        minv = row['min_value_in_window']
        if pd.isna(p5v) or p5v == 0 or pd.isna(minv) or minv == 0:
            return pd.NA
        return p5v / minv

    agg['severity_high'] = agg.apply(sev_high, axis=1)
    agg['severity_low']  = agg.apply(sev_low, axis=1)

    # 5) Keep only rows with any high or low anomalies
    res = agg[(agg['anomalies_count_high'] > 0) | (agg['anomalies_count_low'] > 0)].copy()

    # If still empty, return empty structure
    if res.empty:
        cols = ['cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
                'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
                'p5','severity_low']
        result_df = pd.DataFrame(columns=cols)
        summary_line = "0 distinct trace series had anomalies among the candidate components in the window. Components with anomalies: []"
        result_df, summary_line
    else:
        # sorting: earliest (high if present else low), then severity (high else low) desc
        res['sort_earliest'] = res['earliest_anomaly_ts_high'].fillna(res['earliest_anomaly_ts_low'])
        res['sort_severity'] = res['severity_high'].fillna(res['severity_low'])

        res_sorted = res.sort_values(by=['sort_earliest', 'sort_severity'], ascending=[True, False], na_position='last').head(20)

        # select requested columns and reset index
        result_df = res_sorted[[
            'cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
            'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
            'p5','severity_low'
        ]].reset_index(drop=True)

        # 6) summary: number of distinct trace series (rows) and which components had anomalies
        distinct_series_count = int(result_df.shape[0])
        components_with_anomalies = sorted(result_df['cmdb_id'].unique().tolist())
        summary_line = (f"{distinct_series_count} distinct trace series (among the candidates) had anomalies in the window; "
                        f"components with at least one trace anomaly: {components_with_anomalies}.")

        result_df, summary_line

```
Out[4]:
```
```python
# Recompute trace anomaly summary for the candidate components and return a one-line plain-English summary.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
candidates = ['MG02', 'Mysql01', 'Tomcat03', 'apache02', 'apache01']
win_start_ts = 1615516200
win_end_ts = 1615518000

# Load trace.csv (reuse if already in globals)
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# 1) Full-series p95/p5 per (cmdb_id, trace_name)
pctl = trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter to window and candidate components
trace_win = trace[
    (trace['timestamp'] >= win_start_ts) &
    (trace['timestamp'] <= win_end_ts) &
    (trace['cmdb_id'].isin(candidates))
].copy()

if trace_win.empty:
    summary = ("0 distinct trace series (among MG02, Mysql01, Tomcat03, apache02, apache01) had anomalies "
               f"in the window [{win_start_ts} - {win_end_ts}]. Components with anomalies: [].")
else:
    # 3) Join p95/p5
    trace_win = trace_win.merge(pctl, on=['cmdb_id', 'trace_name'], how='left')

    # flags
    trace_win['is_high'] = trace_win['value'] >= trace_win['p95']
    trace_win['is_low']  = trace_win['value'] <= trace_win['p5']

    # aggregates per trace series
    agg = trace_win.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
        count_in_window = ('timestamp', 'size'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        anomalies_count_high = ('is_high', 'sum'),
        anomalies_count_low  = ('is_low', 'sum'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    ).reset_index()

    # earliest anomaly timestamps
    high_earliest = trace_win[trace_win['is_high']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_ts_high')
    low_earliest  = trace_win[trace_win['is_low']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_ts_low')

    agg = agg.merge(high_earliest, on=['cmdb_id','trace_name'], how='left')
    agg = agg.merge(low_earliest, on=['cmdb_id','trace_name'], how='left')

    # filter to series with any anomalies
    res = agg[(agg['anomalies_count_high'] > 0) | (agg['anomalies_count_low'] > 0)].copy()

    # summary stats
    distinct_series_count = int(res.shape[0])
    components_with_anomalies = sorted(res['cmdb_id'].unique().tolist())

    summary = (f"{distinct_series_count} distinct trace series (among MG02, Mysql01, Tomcat03, apache02, apache01) "
               f"had anomalies in the window [{win_start_ts} - {win_end_ts}]. "
               f"Components with at least one trace anomaly: {components_with_anomalies}.")

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[5]:

# Re-run trace anomaly analysis for the specified candidate components, reusing existing variables where possible.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
candidates = ['MG02', 'Mysql01', 'Tomcat03', 'apache02', 'apache01']
win_start_ts = 1615516200
win_end_ts = 1615518000

# reuse trace if available, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# 1) Full-series p95/p5 per (cmdb_id, trace_name) - reuse if previously computed
if 'trace_pctl' in globals():
    pctl = trace_pctl
else:
    pctl = trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].agg(
        p95=lambda x: x.quantile(0.95),
        p5 = lambda x: x.quantile(0.05)
    ).reset_index()
    trace_pctl = pctl  # cache for reuse

# 2) Filter trace to window and candidate components
trace_win = trace[
    (trace['timestamp'] >= win_start_ts) &
    (trace['timestamp'] <= win_end_ts) &
    (trace['cmdb_id'].isin(candidates))
].copy()

# If no points in window for candidates, return empty result and summary
if trace_win.empty:
    cols = ['cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
            'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
            'p5','severity_low']
    result_df_top = pd.DataFrame(columns=cols)
    summary_line = ("0 distinct trace series (among MG02, Mysql01, Tomcat03, apache02, apache01) "
                    f"had anomalies in the window [{win_start_ts} - {win_end_ts}]. Components with anomalies: [].")
else:
    # 3) Join filtered rows with per-(cmdb_id,trace_name) p95/p5
    trace_win = trace_win.merge(pctl, on=['cmdb_id', 'trace_name'], how='left')

    # 4) Flag high/low anomalies
    trace_win['is_high'] = trace_win['value'] >= trace_win['p95']
    trace_win['is_low']  = trace_win['value'] <= trace_win['p5']

    # Aggregate per (cmdb_id, trace_name)
    agg = trace_win.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
        count_in_window = ('timestamp', 'size'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        anomalies_count_high = ('is_high', 'sum'),
        anomalies_count_low  = ('is_low', 'sum'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    ).reset_index()

    # earliest anomaly timestamps for high and low
    high_earliest = trace_win[trace_win['is_high']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_ts_high')
    low_earliest  = trace_win[trace_win['is_low']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_ts_low')

    agg = agg.merge(high_earliest, on=['cmdb_id','trace_name'], how='left')
    agg = agg.merge(low_earliest, on=['cmdb_id','trace_name'], how='left')

    # ensure integer counts
    agg['anomalies_count_high'] = agg['anomalies_count_high'].astype(int)
    agg['anomalies_count_low']  = agg['anomalies_count_low'].astype(int)
    agg['count_in_window'] = agg['count_in_window'].astype(int)

    # compute severity_high and severity_low with safe checks
    def compute_sev_high(row):
        p95v = row['p95']
        if pd.isna(p95v) or p95v == 0:
            return pd.NA
        return row['max_value_in_window'] / p95v

    def compute_sev_low(row):
        p5v = row['p5']
        minv = row['min_value_in_window']
        if pd.isna(p5v) or p5v == 0 or pd.isna(minv) or minv == 0:
            return pd.NA
        return p5v / minv

    agg['severity_high'] = agg.apply(compute_sev_high, axis=1)
    agg['severity_low']  = agg.apply(compute_sev_low, axis=1)

    # 5) Keep only rows with any high or low anomalies
    res = agg[(agg['anomalies_count_high'] > 0) | (agg['anomalies_count_low'] > 0)].copy()

    # prepare sorting keys: earliest (high if present else low), severity (high else low)
    res['sort_earliest'] = res['earliest_anomaly_ts_high'].fillna(res['earliest_anomaly_ts_low'])
    res['sort_severity'] = res['severity_high'].fillna(res['severity_low'])

    # sort and limit to top 20
    res_sorted = res.sort_values(by=['sort_earliest', 'sort_severity'], ascending=[True, False], na_position='last').head(20)

    # select and order requested columns
    cols = ['cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
            'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
            'p5','severity_low']
    result_df_top = res_sorted[cols].reset_index(drop=True)

    # 6) One-line summary: number of distinct trace series and which components had at least one trace anomaly
    distinct_series_count = int(result_df_top.shape[0])
    components_with_anomalies = sorted(result_df_top['cmdb_id'].unique().tolist())
    summary_line = (f"{distinct_series_count} distinct trace series (among the 5 candidates) had anomalies in the window "
                    f"[{win_start_ts} - {win_end_ts}]. Components with at least one trace anomaly: {components_with_anomalies}.")

# Final compact outputs: result_df_top (<=20 rows) and summary_line
result_df_top, summary_line

```
Out[5]:
```
```python
# Summarize the trace anomaly analysis results in plain English and show a compact top-5 table.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
candidates = ['MG02', 'Mysql01', 'Tomcat03', 'apache02', 'apache01']
win_start_ts = 1615516200
win_end_ts = 1615518000

# Try to reuse result_df_top from previous execution; otherwise create empty placeholders.
try:
    _ = result_df_top
    df = result_df_top.copy()
except NameError:
    df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
        'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
        'p5','severity_low'
    ])

# Build one-line summary
distinct_series_count = int(df.shape[0])
components_with_anomalies = sorted(df['cmdb_id'].unique().tolist())
summary = (f"{distinct_series_count} distinct trace series (among {candidates}) had anomalies in the window "
           f"[{win_start_ts} - {win_end_ts}]. Components with at least one trace anomaly: {components_with_anomalies}.")

# Prepare a compact top-5 table for quick inspection
if not df.empty:
    cols_show = ['cmdb_id','trace_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
                 'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
                 'p5','severity_low']
    top5 = df[cols_show].head(5).copy()

    # convert earliest timestamp columns to readable Asia/Shanghai strings for clarity (keep original ts as well)
    for col in ['earliest_anomaly_ts_high', 'earliest_anomaly_ts_low']:
        if col in top5.columns:
            def fmt_ts(x):
                try:
                    if pd.isna(x):
                        return pd.NA
                    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
                except Exception:
                    return pd.NA
            top5[col + '_sh'] = top5[col].apply(fmt_ts)
else:
    top5 = df.head(0)

# Display the one-line summary and the compact top-5 table
summary, top5
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  count_in_window  anomalies_count_high  earliest_anomaly_ts_high  max_value_in_window        p95  severity_high  anomalies_count_low  earliest_anomaly_ts_low        p5 severity_low
0       MG02    trace.to_dockerB1.duration_mean               30                     2                1615516200             6.114427   3.929498       1.556033                    2               1615517220  0.369485     1.559007
1   Tomcat03        trace.to_MG01.duration_mean               29                     2                1615516200             4.802571   3.103851       1.547294                    2               1615517220  0.325654     7.401232
2   Tomcat03         trace.to_MG01.duration_p95               29                     2                1615516200            26.433000  17.624780       1.499763                    2               1615517700  1.068100       24.275
3       MG02  trace.from_Tomcat02.duration_mean               30                     2                1615516200             4.773708   3.282494       1.454293                    2               1615517220  0.299617     1.176973
4       MG02           trace.self.duration_mean               30                     2                1615516200             5.766851   4.030212       1.430905                    2               1615517520  0.267166     1.117986
5       MG02     trace.to_dockerB2.duration_p95               30                     2                1615516200            32.364950  22.734090       1.423631                    2               1615517220  1.134505     1.055353
6       MG02     trace.to_dockerB1.duration_p95               30                     2                1615516200            36.870750  27.082642       1.361416                    2               1615517220  2.111445      1.96633
7       MG02  trace.from_Tomcat01.duration_mean               30                     2                1615516200             4.335829   3.275914       1.323548                    2               1615517220  0.291663     1.091573
8       MG02    trace.to_dockerB2.duration_mean               30                     2                1615516200             4.727151   3.760807       1.256951                    2               1615517220  0.239543     1.271039
9       MG02   trace.from_Tomcat01.duration_p95               30                     2                1615516200            22.534900  18.098230       1.245144                    2               1615517220  1.160935     1.084986
10      MG02   trace.from_Tomcat03.duration_p95               30                     2                1615516200            26.993050  21.681250       1.244995                    2               1615517220  1.984773     1.752558
11  Tomcat03         trace.to_MG02.duration_p95               30                     2                1615516200            26.993050  21.681250       1.244995                    2               1615517220  1.984773     1.752558
12      MG02  trace.from_Tomcat04.duration_mean               30                     2                1615516200             3.665614   3.085602       1.187974                    2               1615517220  0.376570     1.307711
13      MG02   trace.from_Tomcat04.duration_p95               30                     2                1615516200            20.891000  18.084745       1.155172                    2               1615517220  2.056453     1.009054
14      MG02   trace.from_Tomcat02.duration_p95               30                     2                1615516200            20.977200  18.305805       1.145932                    2               1615517220  1.506185     1.401559
15      MG02  trace.from_dockerA2.duration_mean               30                     2                1615516200            10.585435   9.313230       1.136602                    2               1615516920  0.017831     2.845261
16      MG02  trace.from_dockerB2.duration_mean               30                     2                1615516200            10.524417   9.262404       1.136251                    2               1615517100  0.037170      1.30173
17      MG02  trace.from_Tomcat03.duration_mean               30                     2                1615516200             3.995783   3.569784       1.119335                    2               1615517520  0.306230     1.060283
18  Tomcat03        trace.to_MG02.duration_mean               30                     2                1615516200             3.995783   3.569784       1.119335                    2               1615517520  0.306230     1.060283
19  Tomcat03          trace.from_root.row_count                5                     1                1615516200            40.000000  36.600000       1.092896                    1               1615517760  2.200000          2.2, "20 distinct trace series (among the 5 candidates) had anomalies in the window [1615516200 - 1615518000]. Components with at least one trace anomaly: ['MG02', 'Tomcat03'].")```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse metric if present, else load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# 1) Compute full-series p95 per (cmdb_id, kpi_name) BEFORE filtering
p95_df = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95')

# 2) Filter metric to the inclusive window and MG02
win_start_ts = 1615516200
win_end_ts = 1615518000
mg02_win = metric[
    (metric['cmdb_id'] == 'MG02') &
    (metric['timestamp'] >= win_start_ts) &
    (metric['timestamp'] <= win_end_ts)
].copy()

# If no points for MG02 in window, return empty result and summary
if mg02_win.empty:
    cols = ['kpi_name', 'count_in_window', 'anomalies_count', 'earliest_anomaly_ts',
            'max_value_in_window', 'p95', 'severity']
    mg02_top20 = pd.DataFrame(columns=cols)
    summary_line = "0 distinct MG02 KPIs were anomalous in the window."
else:
    # 3) Join filtered rows with full-series p95
    mg02_win = mg02_win.merge(p95_df[p95_df['cmdb_id'] == 'MG02'][['kpi_name','p95']], on='kpi_name', how='left')

    # 4) Flag anomalies (value >= p95)
    mg02_win['is_anom'] = mg02_win['value'] >= mg02_win['p95']

    # Aggregate per kpi_name
    agg = mg02_win.groupby('kpi_name', dropna=False).agg(
        count_in_window = ('timestamp', 'size'),
        anomalies_count     = ('is_anom', 'sum'),
        max_value_in_window = ('value', 'max'),
        p95                 = ('p95', 'first')
    ).reset_index()

    # earliest anomaly timestamp for each KPI (min timestamp where is_anom True)
    earliest_anom = mg02_win[mg02_win['is_anom']].groupby('kpi_name')['timestamp'].min().reset_index(name='earliest_anomaly_ts')
    agg = agg.merge(earliest_anom, on='kpi_name', how='left')

    # compute severity = max_value_in_window / p95 if p95>0 else NA
    def compute_sev(row):
        p = row['p95']
        if pd.isna(p) or p == 0:
            return pd.NA
        return row['max_value_in_window'] / p

    agg['severity'] = agg.apply(compute_sev, axis=1)

    # keep only KPIs with anomalies_count > 0
    mg02_anom = agg[agg['anomalies_count'] > 0].copy()

    # ensure types and ordering
    mg02_anom['anomalies_count'] = mg02_anom['anomalies_count'].astype(int)
    mg02_anom['count_in_window'] = mg02_anom['count_in_window'].astype(int)
    mg02_anom['earliest_anomaly_ts'] = mg02_anom['earliest_anomaly_ts'].where(mg02_anom['earliest_anomaly_ts'].notna(), pd.NA).astype("Int64")

    #  Sort by severity descending, then earliest_anomaly_ts ascending
    mg02_anom_sorted = mg02_anom.sort_values(by=['severity', 'earliest_anomaly_ts'], ascending=[False, True])

    # select requested columns and limit to top 20
    mg02_top20 = mg02_anom_sorted[[
        'kpi_name', 'count_in_window', 'anomalies_count', 'earliest_anomaly_ts',
        'max_value_in_window', 'p95', 'severity'
    ]].head(20).reset_index(drop=True)

    distinct_count = int(mg02_top20.shape[0])
    summary_line = f"{distinct_count} distinct MG02 KPIs were anomalous in the window [{win_start_ts} - {win_end_ts}]."

# Display compact results: top table and one-line summary
mg02_top20, summary_line

```
Out[6]:
```
```python
# Summarize MG02 metric anomalies in plain English using previously computed mg02_top20.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse mg02_top20 from previous execution if available
if 'mg02_top20' not in globals():
    # Fallback: recompute minimal required data from metric.csv
    metric = pd.read_csv('metric.csv')
    p95_df = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95')
    win_start_ts = 1615516200
    win_end_ts = 1615518000
    mg02_win = metric[
        (metric['cmdb_id'] == 'MG02') &
        (metric['timestamp'] >= win_start_ts) &
        (metric['timestamp'] <= win_end_ts)
    ].copy()
    if mg02_win.empty:
        mg02_top20 = pd.DataFrame(columns=[
            'kpi_name', 'count_in_window', 'anomalies_count', 'earliest_anomaly_ts',
            'max_value_in_window', 'p95', 'severity'
        ])
    else:
        mg02_win = mg02_win.merge(p95_df[p95_df['cmdb_id'] == 'MG02'][['kpi_name','p95']], on='kpi_name', how='left')
        mg02_win['is_anom'] = mg02_win['value'] >= mg02_win['p95']
        agg = mg02_win.groupby('kpi_name', dropna=False).agg(
            count_in_window = ('timestamp', 'size'),
            anomalies_count     = ('is_anom', 'sum'),
            max_value_in_window = ('value', 'max'),
            p95                 = ('p95', 'first')
        ).reset_index()
        earliest_anom = mg02_win[mg02_win['is_anom']].groupby('kpi_name')['timestamp'].min().reset_index(name='earliest_anomaly_ts')
        agg = agg.merge(earliest_anom, on='kpi_name', how='left')
        def compute_sev(row):
            p = row['p95']
            if pd.isna(p) or p == 0:
                return pd.NA
            return row['max_value_in_window'] / p
        agg['severity'] = agg.apply(compute_sev, axis=1)
        mg02_top20 = agg[agg['anomalies_count'] > 0].copy()
        mg02_top20['earliest_anomaly_ts'] = mg02_top20['earliest_anomaly_ts'].astype('Int64')
        mg02_top20 = mg02_top20.sort_values(by=['severity', 'earliest_anomaly_ts'], ascending=[False, True])[
            ['kpi_name', 'count_in_window', 'anomalies_count', 'earliest_anomaly_ts', 'max_value_in_window', 'p95', 'severity']
        ].head(20).reset_index(drop=True)

# Build plain-English summary string
num_anomalous = int(mg02_top20.shape[0])
window_sh_start = pd.to_datetime(1615516200, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
window_sh_end   = pd.to_datetime(1615518000, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

lines = []
lines.append(f"{num_anomalous} distinct MG02 metric series had anomalies in the window {window_sh_start} to {window_sh_end} (Asia/Shanghai).")
if num_anomalous == 0:
    lines.append("No anomalous MG02 KPIs found in the window.")
else:
    lines.append("Top anomalies (by severity):")
    top_n = mg02_top20.head(5)
    for i, row in top_n.iterrows():
        kpi = row['kpi_name']
        sev = row['severity']
        sev_str = f"{float(sev):.3f}" if pd.notna(sev) else "NA"
        ts = row['earliest_anomaly_ts']
        if pd.isna(ts):
            ts_str = "N/A"
        else:
            ts_str = pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        count = int(row['count_in_window'])
        anom_ct = int(row['anomalies_count'])
        maxv = row['max_value_in_window']
        p95 = row['p95']
        lines.append(f"{i+1}. {kpi} — severity={sev_str}, earliest_anomaly={ts_str}, anomalies={anom_ct}, points_in_window={count}, max={maxv}, p95={p95}")

summary = "\n".join(lines)

# Display the single summary string and the compact table (<=20 rows) for reference
summary, mg02_top20
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  count_in_window  anomalies_count  earliest_anomaly_ts  max_value_in_window           p95  severity
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               20                1           1615516500         1.000000e+00  5.000000e-02      20.0
1      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps               11                1           1615516920         1.000000e+00  5.000000e-01       2.0
2      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               10                1           1615517940         7.920000e+00  4.356000e+00  1.818182
3     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               11                1           1615516200         2.000000e-01  1.250000e-01       1.6
4     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               11                1           1615516200         6.000000e+00  3.933350e+00  1.525417
5                              OSLinux-CPU_CPU_CPUWio               18                1           1615516500         1.417900e+00  9.339100e-01  1.518241
6   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...               11                1           1615516920         1.900000e+01  1.550000e+01  1.225806
7     OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec               10                1           1615517340         2.092944e+06  1.724998e+06  1.213302
8      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps               11                1           1615516920         3.000000e+00  2.500000e+00       1.2
9   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...               10                1           1615516320         3.000000e-01  2.550000e-01  1.176471
10                            OSLinux-CPU_CPU_CPULoad               18                1           1615517460         1.760000e+00  1.581500e+00  1.112868
11         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage               20                1           1615517160         6.057970e+01  5.553216e+01  1.090894
12  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...               11                1           1615516920         2.013000e+03  1.891500e+03  1.064235
13          JVM-Operating System_7779_JVM_JVM_CPULoad               20                1           1615517880         4.180100e+00  3.958465e+00   1.05599
14   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite               11                1           1615517100         1.770800e+03  1.677533e+03  1.055597
15                    OSLinux-CPU_CPU-3_SingleCpuUtil                9                1           1615517820         8.481900e+00  8.038540e+00  1.055154
16          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed               31                2           1615517100         1.235908e+09  1.171309e+09  1.055152
17    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps               11                1           1615517100         2.766700e+00  2.625000e+00  1.053981
18                    OSLinux-CPU_CPU-1_SingleCpuUtil                9                1           1615517820         9.007500e+00  8.610380e+00  1.046121
19        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn               10                1           1615517940         2.506000e+03  2.417350e+03  1.036672, '20 distinct MG02 KPIs were anomalous in the window [1615516200 - 1615518000].')```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse loaded DataFrames if present
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

# Parameters
component = 'MG02'
win_start_ts = 1615516200
win_end_ts = 1615518000

# 1) Compute full-series p95 and p5 per (cmdb_id, log_name) BEFORE filtering
log_pctl = log.groupby(['cmdb_id', 'log_name'], dropna=False)['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter log.csv to MG02 and the inclusive window
log_win = log[
    (log['cmdb_id'] == component) &
    (log['timestamp'] >= win_start_ts) &
    (log['timestamp'] <= win_end_ts)
].copy()

# Prepare default empty outputs
cols_log = ['log_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
            'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
            'p5','severity_low']
log_result_top20 = pd.DataFrame(columns=cols_log)
log_summary = f"0 distinct {component} log series were anomalous in the window."

if not log_win.empty:
    # 3) Join filtered rows with per-(cmdb_id,log_name) p95/p5
    log_win = log_win.merge(log_pctl[log_pctl['cmdb_id'] == component], on=['cmdb_id','log_name'], how='left')

    # 4) Flag high/low anomalies
    log_win['is_high'] = log_win['value'] >= log_win['p95']
    log_win['is_low']  = log_win['value'] <= log_win['p5']

    # Aggregate per log_name
    agg = log_win.groupby('log_name', dropna=False).agg(
        count_in_window = ('timestamp', 'size'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        anomalies_count_high = ('is_high', 'sum'),
        anomalies_count_low  = ('is_low', 'sum'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    ).reset_index()

    # Earliest anomaly timestamps
    high_earliest = log_win[log_win['is_high']].groupby('log_name')['timestamp'].min().reset_index(name='earliest_anomaly_ts_high')
    low_earliest  = log_win[log_win['is_low']].groupby('log_name')['timestamp'].min().reset_index(name='earliest_anomaly_ts_low')

    agg = agg.merge(high_earliest, on='log_name', how='left')
    agg = agg.merge(low_earliest, on='log_name', how='left')

    # ensure integer counts
    agg['anomalies_count_high'] = agg['anomalies_count_high'].astype(int)
    agg['anomalies_count_low']  = agg['anomalies_count_low'].astype(int)
    agg['count_in_window'] = agg['count_in_window'].astype(int)

    # compute severities with safety checks
    def compute_sev_high(row):
        p95v = row['p95']
        if pd.isna(p95v) or p95v == 0:
            return pd.NA
        return row['max_value_in_window'] / p95v

    def compute_sev_low(row):
        p5v = row['p5']
        minv = row['min_value_in_window']
        if pd.isna(p5v) or p5v == 0 or pd.isna(minv) or minv == 0:
            return pd.NA
        return p5v / minv

    agg['severity_high'] = agg.apply(compute_sev_high, axis=1)
    agg['severity_low']  = agg.apply(compute_sev_low, axis=1)

    # 4) Keep only rows where high or low anomalies exist
    res = agg[(agg['anomalies_count_high'] > 0) | (agg['anomalies_count_low'] > 0)].copy()

    if not res.empty:
        # 5) Sorting: earliest anomaly (high preferred), then severity (high preferred)
        res['sort_earliest'] = res['earliest_anomaly_ts_high'].fillna(res['earliest_anomaly_ts_low'])
        res['sort_severity'] = res['severity_high'].fillna(res['severity_low'])

        res_sorted = res.sort_values(by=['sort_earliest', 'sort_severity'], ascending=[True, False], na_position='last').head(20)

        # select output columns
        log_result_top20 = res_sorted[[
            'log_name','count_in_window','anomalies_count_high','earliest_anomaly_ts_high',
            'max_value_in_window','p95','severity_high','anomalies_count_low','earliest_anomaly_ts_low',
            'p5','severity_low'
        ]].reset_index(drop=True)

        num_log_series = int(log_result_top20.shape[0])
        log_summary = f"{num_log_series} distinct {component} log series were anomalous in the window [{win_start_ts} - {win_end_ts}]."
    else:
        log_result_top20 = pd.DataFrame(columns=cols_log)
        log_summary = f"0 distinct {component} log series were anomalous in the window."

# 2) ERROR LOGS: filter error_logs.csv for MG02 in window
err_win = error_logs[
    (error_logs['cmdb_id'] == component) &
    (error_logs['timestamp'] >= win_start_ts) &
    (error_logs['timestamp'] <= win_end_ts)
].copy()

if err_win.empty:
    error_summary_line = f"No error_logs for {component} in the window [{win_start_ts} - {win_end_ts}]."
    error_summary_df = pd.DataFrame(columns=['message','count'])
    total_error_rows_in_window = 0
    earliest_timestamp_in_window = pd.NA
    latest_timestamp_in_window = pd.NA
else:
    total_error_rows_in_window = int(err_win.shape[0])
    earliest_timestamp_in_window = int(err_win['timestamp'].min())
    latest_timestamp_in_window = int(err_win['timestamp'].max())

    # Top up to 20 distinct messages by count desc
    msg_counts = err_win.groupby('message', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
    error_summary_df = msg_counts[['message','count']]

    error_summary_line = (f"{total_error_rows_in_window} error log rows for {component} in the window; "
                          f"earliest_ts={earliest_timestamp_in_window}, latest_ts={latest_timestamp_in_window}. "
                          f"Top messages (up to 20) shown in table.")

# Final compact outputs (DataFrames and summary strings)
log_result_top20, log_summary, error_summary_df, error_summary_line

```
Out[7]:
```
```python
# Summarize findings for MG02 logs and error messages in the window using prior analysis results.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
win_start_ts = 1615516200
win_end_ts = 1615518000
win_start_sh = pd.to_datetime(win_start_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
win_end_sh = pd.to_datetime(win_end_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

summary = (
    f"MG02 logs & error messages analysis for window {win_start_sh} to {win_end_sh}:\n"
    f"- Log series: No MG02 log series showed anomalies in the window when judged against full-series P95/P5 thresholds.\n"
    f"- Error logs: No error_log entries for MG02 were found in the window.\n\n"
    f"Conclusion: There is no corroborating evidence in MG02 logs or error messages to confirm a root cause or an exact occurrence time within this window. "
    f"Investigators should rely on metric/trace anomalies (which did show MG02 issues) for root-cause inference and then search logs around the indicated metric/trace timestamps for additional corroboration."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [log_name, count_in_window, anomalies_count_high, earliest_anomaly_ts_high, max_value_in_window, p95, severity_high, anomalies_count_low, earliest_anomaly_ts_low, p5, severity_low]
Index: [], '0 distinct MG02 log series were anomalous in the window.', Empty DataFrame
Columns: [message, count]
Index: [], 'No error_logs for MG02 in the window [1615516200 - 1615518000].')```
```